In [2]:
%%writefile Dockerfile
# Usa una imagen oficial de Python liviana
FROM python:3.11-slim

# Establece el directorio de trabajo
WORKDIR /app

# Copia e instala dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copia el script de Python al contenedor
COPY multicloud_batch_worker.py .

# Comando por defecto para ejecutar el worker
CMD ["python", "multicloud_batch_worker.py"]

Writing Dockerfile


In [5]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Fleet Multicloud Worker (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu script de telemetría no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Ejecutar Prueba del Procesamiento de Telemetría (Mock Enterprise)
      run: |
        # Ejecutamos el script simulando el entorno real de AWS/Azure con Moto
        export SIMULACION=False
        export AWS_INPUT_BUCKET=dev-fleet-raw-data
        export AWS_INPUT_KEY=daily_telemetry.csv
        export AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
        export AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
        export AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
        
        python multicloud_batch_worker.py

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos el registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker del Fleet Worker
      run: |
        # Construimos la imagen de Docker usando el Dockerfile del Worker
        docker build -t fleet-multicloud-worker:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local
        docker tag fleet-multicloud-worker:latest localhost:5000/fleet-multicloud-worker-repo:latest
        
        # Subimos la imagen para simular la publicación en producción
        docker push localhost:5000/fleet-multicloud-worker-repo:latest
        echo "¡Imagen de Fleet Worker subida con éxito al registro simulado!"

Writing .github/workflows/ci-cd-pipeline.yml


In [4]:
# Crea las carpetas ocultas necesarias para que GitHub Actions encuentre el pipeline
import os
os.makedirs(".github/workflows", exist_ok=True)
print("¡Carpetas .github/workflows creadas con éxito! 🎉")

¡Carpetas .github/workflows creadas con éxito! 🎉


In [6]:
# Añadimos los archivos específicos para no subir por accidente tu Notebook (.ipynb)
!git add .github/workflows/ci-cd-pipeline.yml
!git add multicloud_batch_worker.py
!git add Dockerfile
!git add requirements.txt
!git add docker-compose.yml

# Creamos el commit
!git commit -m "feat: setup clean pipeline for multicloud batch worker with local registry"

# Hacemos el push a GitHub
!git push origin main

[main 8472a7b] feat: setup clean pipeline for multicloud batch worker with local registry
 5 files changed, 270 insertions(+)
 create mode 100644 ejercicio_2/.github/workflows/ci-cd-pipeline.yml
 create mode 100644 ejercicio_2/Dockerfile
 create mode 100644 ejercicio_2/docker-compose.yml
 create mode 100644 ejercicio_2/multicloud_batch_worker.py
 create mode 100644 ejercicio_2/requirements.txt
Counting objects: 10, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (8/8), done.
Writing objects: 100% (10/10), 5.01 KiB | 5.01 MiB/s, done.
Total 10 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/leopinzon75/architect-python-api.git
   d9915ea..8472a7b  main -> main


In [7]:
# 1. Movemos el archivo de workflows a la raíz real del repositorio
!git mv ejercicio_2/.github/workflows/ci-cd-pipeline.yml .github/workflows/ci-cd-pipeline.yml

# 2. Confirmamos el movimiento y hacemos el push
!git commit -m "chore: move workflow file to the repository root so GitHub Actions can find it"
!git push origin main

fatal: bad source, source=ejercicio_2/ejercicio_2/.github/workflows/ci-cd-pipeline.yml, destination=ejercicio_2/.github/workflows/ci-cd-pipeline.yml
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
	../.ipynb_checkpoints/
	.ipynb_checkpoints/
	ejer_2.ipynb
	"../fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present
Everything up-to-date


In [8]:
# 1. Movemos el archivo desde tu ubicación actual en ejercicio_2 hacia la raíz del repositorio (../)
!git mv .github/workflows/ci-cd-pipeline.yml ../.github/workflows/ci-cd-pipeline.yml

# 2. Registramos el cambio con un commit
!git commit -m "chore: relocate workflow to repo root from subfolder"

# 3. Lo subimos a GitHub
!git push origin main

fatal: destination exists, source=ejercicio_2/.github/workflows/ci-cd-pipeline.yml, destination=.github/workflows/ci-cd-pipeline.yml
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
	../.ipynb_checkpoints/
	.ipynb_checkpoints/
	ejer_2.ipynb
	"../fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present
Everything up-to-date


In [9]:
# 1. Forzamos el movimiento para sobreescribir el archivo de la raíz (../) con el nuevo
!git mv -f .github/workflows/ci-cd-pipeline.yml ../.github/workflows/ci-cd-pipeline.yml

# 2. Confirmamos el reemplazo
!git commit -m "chore: replace root pipeline with the new fleet worker workflow"

# 3. Lo subimos a GitHub
!git push origin main

[main 4e3afeb] chore: replace root pipeline with the new fleet worker workflow
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename {ejercicio_2/.github => .github}/workflows/ci-cd-pipeline.yml (100%)
Counting objects: 3, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 454 bytes | 454.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/leopinzon75/architect-python-api.git
   8472a7b..4e3afeb  main -> main


In [10]:
%%writefile ../.github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Fleet Multicloud Worker (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      # Le indicamos que busque el requirements.txt dentro de ejercicio_2
      working-directory: ./ejercicio_2
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      working-directory: ./ejercicio_2
      run: |
        # Validamos que tu script de telemetría no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Ejecutar Prueba del Procesamiento de Telemetría (Mock Enterprise)
      # Nos paramos en ejercicio_2 para que encuentre el script multicloud_batch_worker.py
      working-directory: ./ejercicio_2
      run: |
        export SIMULACION=False
        export AWS_INPUT_BUCKET=dev-fleet-raw-data
        export AWS_INPUT_KEY=daily_telemetry.csv
        export AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
        export AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
        export AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
        
        python multicloud_batch_worker.py

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos el registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker del Fleet Worker
      # Nos paramos en ejercicio_2 para usar el Dockerfile correcto
      working-directory: ./ejercicio_2
      run: |
        docker build -t fleet-multicloud-worker:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local
        docker tag fleet-multicloud-worker:latest localhost:5000/fleet-multicloud-worker-repo:latest
        
        # Subimos la imagen para simular la publicación en producción
        docker push localhost:5000/fleet-multicloud-worker-repo:latest
        echo "¡Imagen de Fleet Worker subida con éxito al registro simulado!"

Overwriting ../.github/workflows/ci-cd-pipeline.yml


In [11]:
# Guardamos los cambios del pipeline que está en la raíz (../)
!git add ../.github/workflows/ci-cd-pipeline.yml
!git commit -m "fix: set working directory to ejercicio_2 in pipeline steps"
!git push origin main    

[main cc1cbad] fix: set working directory to ejercicio_2 in pipeline steps
 1 file changed, 7 insertions(+), 2 deletions(-)
Counting objects: 5, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 619 bytes | 619.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/leopinzon75/architect-python-api.git
   4e3afeb..cc1cbad  main -> main


In [12]:
%%writefile ../.github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Fleet Multicloud Worker (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      working-directory: ./ejercicio_2
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      working-directory: ./ejercicio_2
      run: |
        # Validamos que tu script de telemetría no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Ejecutar Prueba del Procesamiento de Telemetría (Mock Enterprise)
      working-directory: ./ejercicio_2
      run: |
        # CAMBIAMOS A MODO SIMULACIÓN VERDADERO (3 FILAS EN MEMORIA)
        export SIMULACION=True
        export AWS_INPUT_BUCKET=dev-fleet-raw-data
        export AWS_INPUT_KEY=daily_telemetry.csv
        export AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
        export AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
        export AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
        
        python multicloud_batch_worker.py

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos el registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker del Fleet Worker
      working-directory: ./ejercicio_2
      run: |
        docker build -t fleet-multicloud-worker:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local
        docker tag fleet-multicloud-worker:latest localhost:5000/fleet-multicloud-worker-repo:latest
        
        # Subimos la imagen para simular la publicación en producción
        docker push localhost:5000/fleet-multicloud-worker-repo:latest
        echo "¡Imagen de Fleet Worker subida con éxito al registro simulado!"

Overwriting ../.github/workflows/ci-cd-pipeline.yml


In [13]:
# Guardamos los cambios y los empujamos a GitHub
!git add ../.github/workflows/ci-cd-pipeline.yml
!git commit -m "test: switch pipeline to SIMULACION=True to test in-memory branch"
!git push origin main

[main ac06f65] test: switch pipeline to SIMULACION=True to test in-memory branch
 1 file changed, 2 insertions(+), 4 deletions(-)
Counting objects: 5, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 546 bytes | 546.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/leopinzon75/architect-python-api.git
   cc1cbad..ac06f65  main -> main


In [14]:
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

import os
from pathlib import Path

# =========================================================
# CAPTURA DE VARIABLES DE ENTORNO (EL PUENTE CON DOCKER)
# =========================================================

# os.environ.get busca la variable en Docker. Si no la encuentra, usa el valor por defecto.
# IMPORTANTE: Docker todo lo manda como texto ("True" o "False").
# Por eso lo comparamos con el texto "True" para convertirlo en un Booleano real de Python.
SIMULATION_ENV = os.environ.get("SIMULACION", "True")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"

# Capturamos el resto de las variables que configuraste en el YAML
AWS_BUCKET = os.environ.get("AWS_INPUT_BUCKET", "fleet-raw-data")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "raw_telemetry.csv")
AZURE_CONTAINER = os.environ.get("AZURE_OUTPUT_CONTAINER", "fleet-clean-alerts")

print(f"🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
print(f"🎛️  Modo de Simulación: {'ACTIVADO (3 filas)' if SIMULATION_MODE else 'DESACTIVADO (1,000 filas con AWS S3)'}")
####################
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")
    
    SIMULACION = os.getenv("SIMULACION", "True").upper() == "TRUE"
    
    s3, azure_container_client = None, None
    
    if not SIMULACION:
        print("⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...")
        
        # 1. ACTIVAR EL MOCK DE MOTO PARA INTERCEPTAR AWS
        from moto import mock_aws
        
        # Iniciamos el contexto simulado de AWS
        mock_context = mock_aws()
        mock_context.start()
        
        # Creamos el cliente S3 virtual apuntando al entorno local en memoria
        s3 = boto3.client("s3", region_name="us-east-1")
        
        # Creamos el bucket virtual y le inyectamos los 1,000 registros ficticios de telemetría
        s3.create_bucket(Bucket=AWS_BUCKET)
        
        # Fabricamos un lote masivo de telemetría simulada (1,000 líneas) para la prueba
        headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
        bulk_data = headers
        for i in range(1, 1001):
            code = "P0300" if i % 50 == 0 else "P0171" if i % 75 == 0 else "NONE"
            bulk_data += f"{datetime.now()},VIN-{100000 + i},2100,90,{code}\n"
            
        # Subimos el lote masivo al AWS S3 simulado en memoria
        s3.put_object(Bucket=AWS_BUCKET, Key=AWS_KEY, Body=bulk_data.encode("utf-8"))
        print(f"📦 [S3 Mock] Archivo '{AWS_KEY}' con 1,000 registros cargado con éxito en el Bucket virtual.")
        
        # Simulador local para Azure (Mock)
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                print(f"☁️  [Local Container] Reporte '{name}' interceptado exitosamente en simulación local.")
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        print("⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.")
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")

🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🎛️  Modo de Simulación: ACTIVADO (3 filas)
🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.
📥 Connecting to S3 Intake Stream...
🛠️  [Simulación] Fabricando flujo de datos local en memoria...
💾 [Auditoría Local] Réplica cruda de entrada guardada en: data/archive/input/raw_s3_mirror_20260714_203147.csv
📤 Preparing handoff for 2 processed records...
☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.
💾 [Auditoría Local] Reporte filtrado guardado en: data/archive/output/clean_alerts_20260714_203147.csv
🏁 Batch complete! Multi-cloud architectural loop completed successfully.


In [15]:
%%writefile ../.github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Fleet Multicloud Worker (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      working-directory: ./ejercicio_2
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      working-directory: ./ejercicio_2
      run: |
        # Validamos que tu script de telemetría no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Ejecutar Prueba del Procesamiento de Telemetría (Mock Enterprise)
      working-directory: ./ejercicio_2
      run: |
        # VOLVEMOS AL MODO ENTERPRISE ROBUSTO (1,000 REGISTROS EN MOTO S3)
        export SIMULACION=False
        export AWS_INPUT_BUCKET=dev-fleet-raw-data
        export AWS_INPUT_KEY=daily_telemetry.csv
        export AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
        export AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
        export AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
        
        python multicloud_batch_worker.py

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos el registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker del Fleet Worker
      working-directory: ./ejercicio_2
      run: |
        docker build -t fleet-multicloud-worker:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local
        docker tag fleet-multicloud-worker:latest localhost:5000/fleet-multicloud-worker-repo:latest
        
        # Subimos la imagen para simular la publicación en producción
        docker push localhost:5000/fleet-multicloud-worker-repo:latest
        echo "¡Imagen de Fleet Worker subida con éxito al registro simulado!"

Overwriting ../.github/workflows/ci-cd-pipeline.yml


In [16]:
# Subimos la versión robusta final
!git add ../.github/workflows/ci-cd-pipeline.yml
!git commit -m "style: restore pipeline to full mock enterprise mode (1000 records)"
!git push origin main

[main 00b10eb] style: restore pipeline to full mock enterprise mode (1000 records)
 1 file changed, 2 insertions(+), 2 deletions(-)
Counting objects: 5, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 525 bytes | 525.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/leopinzon75/architect-python-api.git
   ac06f65..00b10eb  main -> main


In [17]:
%%writefile ../.github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Fleet Multicloud Worker (Full CD)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-publish:
    runs-on: ubuntu-latest
    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      working-directory: ./ejercicio_2
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      working-directory: ./ejercicio_2
      run: |
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Ejecutar Prueba de Telemetría (1,000 registros con Moto S3)
      working-directory: ./ejercicio_2
      run: |
        export SIMULACION=False
        export AWS_INPUT_BUCKET=dev-fleet-raw-data
        export AWS_INPUT_KEY=daily_telemetry.csv
        export AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
        export AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
        export AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
        
        python multicloud_batch_worker.py

    - name: Iniciar Sesión de forma Segura en Docker Hub
      uses: docker/login-action@v3
      with:
        username: ${{ secrets.DOCKER_USERNAME }}
        password: ${{ secrets.DOCKER_PASSWORD }}

    - name: Construir y Empujar Imagen Real a Docker Hub
      working-directory: ./ejercicio_2
      run: |
        docker build -t ${{ secrets.DOCKER_USERNAME }}/fleet-multicloud-worker:latest .
        docker push ${{ secrets.DOCKER_USERNAME }}/fleet-multicloud-worker:latest
        echo "¡Imagen de producción empujada exitosamente a Docker Hub!"

  deploy-to-production:
    runs-on: ubuntu-latest
    needs: build-test-and-publish
    steps:
    - name: Simular Despliegue Seguro via SSH en Servidor de Producción
      run: |
        echo "🔌 Conectándose via SSH al servidor de producción: prod-srv-mobybunny.internal..."
        echo "🔑 Autenticación exitosa usando llaves SSH seguras."
        echo "📥 Descargando la última versión del contenedor desde Docker Hub..."
        echo "🐳 Comando: docker pull ${{ secrets.DOCKER_USERNAME }}/fleet-multicloud-worker:latest"
        
        echo "🔄 Deteniendo de forma segura el contenedor anterior..."
        echo "🐳 Comando: docker compose down"
        
        echo "🚀 Levantando la nueva versión del microservicio con Docker Compose..."
        echo "🐳 Comando: docker compose up -d"
        
        echo "🛡️ Realizando pruebas de salud de producción (Health Checks)..."
        echo "✅ ¡Despliegue Continuo (CD) completado con éxito y sin interrupciones!"

Overwriting ../.github/workflows/ci-cd-pipeline.yml


In [18]:
!git add ../.github/workflows/ci-cd-pipeline.yml
!git commit -m "feat: complete CI/CD pipeline with Docker Hub push and simulated SSH deploy"
!git push origin main

[main 83a93fb] feat: complete CI/CD pipeline with Docker Hub push and simulated SSH deploy
 1 file changed, 29 insertions(+), 21 deletions(-)
Counting objects: 5, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 1.57 KiB | 1.57 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/leopinzon75/architect-python-api.git
   00b10eb..83a93fb  main -> main


In [20]:
# 1. Detener cualquier contenedor que haya quedado corriendo en segundo plano
!docker compose down

# 2. Limpiar imágenes huérfanas, contenedores apagados y redes que ya no uses
!docker system prune -f

Deleted Containers:
49248da7f823f01800d5915655c6cd32a4c596fbca76a55b74792dd9186adb52
258b84cca16b446ebf37de026463c5563805cdd539e0a8d6bc2064a782ad9cbd
6620ddd8119972afc2f61180a1fd29f2b577bcf78becd381a142b8e3afd2baf5
8d022894a8ddca1d78b7968079149d7a61a743610371d195efeb3402c424761f
46bbc448e75b0781f1ae87f9fdedfe9bd8e63c48438d430d7c577704ef6301a3
6a084beb34683582c9ce1df4c03e700db1978f6bf86d53044a204e216dbf1116
dab44cd06272f0439e88943452db451a49cc2f150505e1e4cc43c0b7ed718b96
f28898516eb5cd83b02a409193c650520248f2398c36304b5cb27cdb336b7be6
057b9f780a0c80e9710cd932e016461ccbcba45bdaf9861fa45f54df935aca26
5ab386e4978334c2edb216100a8528d1ae49fe588cc9458a0301dfc864094cd7
dd8ac917caa01eacaaf077d7864297b82a6951530b713145132795285780c224
c35b811b6485102d22bf0903618eac08cefaba0177be2193c0e935da84aac313
0c8364655bcd73d07743264decf8f7bfe00a1f64a193e7ab38d2047d5e951157
f51a242141670353921776845f061974081778eac41c00c910fc6971e0ce2308
821761f9103058ac22705f784f0cc95bd384678d23f6883694e7c8a5990d4d97
6e18d